# Optional lab: Oxford-IIIT Pet segmentation with U-Net
Use Python with torch, torchvision, numpy, pillow, and matplotlib. This notebook implements binary pet segmentation, reproducible 80/10/10 splits, synchronized augmentation, best-validation checkpoint selection, and held-out Dice/IoU evaluation.

Set `PET_DATASET_PATH` to a folder containing `images/` and `annotations/trimaps/`, or attach the course's Kaggle dataset. With `DOWNLOAD=True`, torchvision fetches the original Oxford dataset. `SMOKE_TEST=True` instead runs explicitly synthetic tensors to check model shapes and optimization; it does not measure pet segmentation quality.


**Run package:** execute the full notebook. Written observations mentioning earlier CPU checks are reference observations; send your outputs back so they can be updated for your run.


**Name:** Utkrisht Suman  
**Roll number:** A24ARIU0001

Prepared for real-data training. Prior synthetic checks do not establish segmentation quality. The run below saves a best validation checkpoint and measures the held-out test set.

In [ ]:
import os, random, copy
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import functional as TF, InterpolationMode
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIZE, BATCH_SIZE, EPOCHS = 128, 16, 10
SMOKE_TEST = False
DOWNLOAD = True
DATA_ROOT = Path(os.environ.get("PET_DATASET_PATH", "data"))
OUT = Path("unet_outputs")
OUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT = OUT / "unet_best_UTKRISHT.pt"
print("device", DEVICE, "smoke", SMOKE_TEST)


## Data preparation
Oxford trimaps encode pet interior as 1, background as 2, and uncertain boundary as 3. Here boundary counts as foreground: `(trimap != 2)`. This avoids discarding fine boundary pixels but can overestimate foreground. Masks use nearest-neighbour resizing. Images use bilinear resizing. Train-only rotation and horizontal flip share their sampled geometry between image and mask; brightness modifies only images. An image-level split keeps each image in exactly one partition.


In [ ]:
def find_pairs(root):
    candidates = [root]
    if Path("/kaggle/input").exists():
        candidates.extend(
            p.parent for p in Path("/kaggle/input").rglob("trimaps") if p.is_dir()
        )
    for candidate in candidates:
        for masks in [
            candidate / "annotations" / "trimaps",
            candidate / "trimaps",
            candidate / "oxford-iiit-pet" / "annotations" / "trimaps",
        ]:
            if not masks.is_dir():
                continue
            for images in [masks.parent.parent / "images", candidate / "images"]:
                if images.is_dir():
                    pairs = [
                        (image, masks / (image.stem + ".png"))
                        for image in sorted(images.glob("*.jpg"))
                    ]
                    pairs = [(a, b) for a, b in pairs if b.is_file()]
                    if pairs:
                        return pairs
    raise FileNotFoundError(
        "Attach Oxford-IIIT Pet images + trimaps or set PET_DATASET_PATH / DOWNLOAD=True"
    )


class PetDataset(Dataset):
    def __init__(self, pairs, augment=False):
        self.pairs, self.augment = pairs, augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        a, b = self.pairs[index]
        with Image.open(a) as image:
            image = image.convert("RGB")
        with Image.open(b) as mask:
            mask = mask.copy()
        assert image.size == mask.size, (a, image.size, mask.size)
        image = TF.resize(image, [SIZE, SIZE], interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, [SIZE, SIZE], interpolation=InterpolationMode.NEAREST)
        if self.augment:
            angle = random.uniform(-10, 10)
            image = TF.rotate(
                image, angle, interpolation=InterpolationMode.BILINEAR, fill=0
            )
            mask = TF.rotate(
                mask, angle, interpolation=InterpolationMode.NEAREST, fill=2
            )
            if random.random() < 0.5:
                image, mask = TF.hflip(image), TF.hflip(mask)
            image = TF.adjust_brightness(image, random.uniform(0.85, 1.15))
        image = TF.to_tensor(image)
        mask = torch.from_numpy((np.array(mask) != 2).astype("float32")).unsqueeze(0)
        assert mask.unique().numel() <= 2
        return image, mask


if not SMOKE_TEST:
    if DOWNLOAD:
        datasets.OxfordIIITPet(
            str(DATA_ROOT), download=True, target_types="segmentation"
        )
    pairs = find_pairs(DATA_ROOT)
    order = np.random.default_rng(SEED).permutation(len(pairs))
    first, second = int(0.8 * len(pairs)), int(0.9 * len(pairs))
    train_set = PetDataset([pairs[i] for i in order[:first]], True)
    valid_set = PetDataset([pairs[i] for i in order[first:second]])
    test_set = PetDataset([pairs[i] for i in order[second:]])
    assert min(len(train_set), len(valid_set), len(test_set)) >= 8
    print("train/validation/test", len(train_set), len(valid_set), len(test_set))
    fig, axes = plt.subplots(5, 2, figsize=(7, 12))
    for i in range(5):
        image, mask = valid_set[i]
        axes[i, 0].imshow(image.permute(1, 2, 0))
        axes[i, 1].imshow(mask[0], cmap="gray", vmin=0, vmax=1)
        for ax in axes[i]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()
    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
        num_workers=0,
    )
    valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)


In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, a, b):
        super().__init__(
            nn.Conv2d(a, b, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(b, b, 3, padding=1),
            nn.ReLU(inplace=True),
        )


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1, self.enc2, self.enc3 = (
            DoubleConv(3, 32),
            DoubleConv(32, 64),
            DoubleConv(64, 128),
        )
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(128, 256)
        self.up3, self.up2, self.up1 = (
            nn.ConvTranspose2d(256, 128, 2, 2),
            nn.ConvTranspose2d(128, 64, 2, 2),
            nn.ConvTranspose2d(64, 32, 2, 2),
        )
        self.dec3, self.dec2, self.dec1 = (
            DoubleConv(256, 128),
            DoubleConv(128, 64),
            DoubleConv(64, 32),
        )
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        a = self.enc1(x)
        b = self.enc2(self.pool(a))
        c = self.enc3(self.pool(b))
        z = self.bottleneck(self.pool(c))
        d = self.dec3(torch.cat([self.up3(z), c], 1))
        e = self.dec2(torch.cat([self.up2(d), b], 1))
        f = self.dec1(torch.cat([self.up1(e), a], 1))
        return self.out(f)


model = UNet().to(DEVICE)
shapes = {}
handles = [
    getattr(model, name).register_forward_hook(
        lambda module, inputs, output, name=name: shapes.update(
            {name: tuple(output.shape)}
        )
    )
    for name in ["enc1", "enc2", "enc3", "bottleneck", "dec3", "dec2", "dec1", "out"]
]
with torch.no_grad():
    result = model(torch.rand(2, 3, SIZE, SIZE, device=DEVICE))
for handle in handles:
    handle.remove()
assert result.shape == (2, 1, SIZE, SIZE)
print(shapes)
print("parameters", sum(p.numel() for p in model.parameters()))


## Receptive field
Each 3×3 convolution combines adjacent features. Pooling by two doubles the spacing between adjacent features measured in original input pixels, so subsequent convolutions see a larger input region. For this encoder, receptive field sizes after the first block, second block, third block, and bottleneck are 5, 14, 32, and 68 pixels respectively (assuming the usual 2×2 stride-two pooling). Skip concatenations restore fine spatial features that pooling can lose, helping reconstruct boundaries and narrow parts.

## Training and evaluation
The model produces logits. Training uses BCEWithLogitsLoss plus soft Dice loss. Thresholded metrics below aggregate intersections and areas over the complete split (micro Dice/IoU). The best model is chosen using validation Dice; test is used once after that choice. Pixel accuracy is avoided because background can dominate.


In [ ]:
def loss_fn(logits, target):
    probs = logits.sigmoid()
    dims = (1, 2, 3)
    dice = (2 * (probs * target).sum(dims) + 1e-6) / (
        probs.sum(dims) + target.sum(dims) + 1e-6
    )
    return (
        nn.functional.binary_cross_entropy_with_logits(logits, target) + 1 - dice.mean()
    )


def epoch(loader, optimizer=None):
    model.train(optimizer is not None)
    total, seen, intersection, predicted, truth = 0.0, 0, 0.0, 0.0, 0.0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        with torch.set_grad_enabled(optimizer is not None):
            logits = model(images)
            loss = loss_fn(logits, targets)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        total += loss.item() * len(images)
        seen += len(images)
        pred = logits.detach().sigmoid() >= 0.5
        intersection += (pred * targets).sum().item()
        predicted += pred.sum().item()
        truth += targets.sum().item()
    return dict(
        loss=total / seen,
        dice=(2 * intersection + 1e-6) / (predicted + truth + 1e-6),
        iou=(intersection + 1e-6) / (predicted + truth - intersection + 1e-6),
    )


if SMOKE_TEST:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    x = torch.rand(2, 3, SIZE, SIZE, device=DEVICE)
    y = (torch.rand(2, 1, SIZE, SIZE, device=DEVICE) > 0.5).float()
    before = model.out.weight.detach().clone()
    loss = loss_fn(model(x), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    assert torch.isfinite(loss) and not torch.equal(before, model.out.weight)
    print("Synthetic shape/backprop checks passed. No pet quality metrics measured.")
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = []
    best = -1
    for index in range(EPOCHS):
        training = epoch(train_loader, optimizer)
        validation = epoch(valid_loader)
        history.append((training, validation))
        print(index + 1, training, validation, flush=True)
        if validation["dice"] > best:
            best = validation["dice"]
            torch.save(model.state_dict(), CHECKPOINT)
    model.load_state_dict(
        torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
    )
    test_metrics = epoch(test_loader)
    print("Held-out test (best validation checkpoint)", test_metrics)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for key, ax in zip(["loss", "dice"], axes):
        for split, label in [(0, "train"), (1, "validation")]:
            ax.plot(
                range(1, len(history) + 1),
                [row[split][key] for row in history],
                label=label,
            )
        ax.set(xlabel="Epoch", ylabel=key)
        ax.legend()
    plt.tight_layout()
    plt.show()
    model.eval()
    fig, axes = plt.subplots(8, 3, figsize=(9, 20))
    failures = []
    with torch.no_grad():
        for i in range(8):
            image, mask = test_set[i]
            pred = (model(image[None].to(DEVICE)).sigmoid()[0].cpu() >= 0.5).float()
            dice = (2 * (pred * mask).sum() + 1e-6) / (pred.sum() + mask.sum() + 1e-6)
            failures.append((dice.item(), i))
            axes[i, 0].imshow(image.permute(1, 2, 0))
            axes[i, 1].imshow(mask[0], cmap="gray")
            axes[i, 2].imshow(pred[0], cmap="gray")
            axes[i, 2].set_title(f"Dice {dice:.3f}")
            for ax in axes[i]:
                ax.axis("off")
    axes[0, 0].set_title("Input")
    axes[0, 1].set_title("Target")
    plt.tight_layout()
    plt.show()
    print("Two lowest-Dice examples among displayed test cases:", sorted(failures)[:2])

if not SMOKE_TEST:
    import json
    (OUT / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
    (OUT / "history.json").write_text(json.dumps(history, indent=2))


## Failure analysis
Use the two lowest-Dice displayed examples identified by the cell above to inspect false foreground and missing pet regions. Candidate mechanisms include similar pet/background colours, occlusion, thin legs or tails lost at 128×128 resolution, and ambiguous trimap boundaries. These are hypotheses to check against actual predictions; no particular failure is claimed before training. Compare predicted masks with the target to distinguish boundary errors from missing entire regions. Report both the measured Dice/IoU and these visual findings; high aggregate overlap can hide errors on small objects.

The optional extensions are deliberately separate from the core model: three-class trimaps, skip ablation, dilated bottlenecks, and BCE-versus-Dice comparison each need a fresh controlled experiment.


## Download the result ZIP
The next cell collects small outputs only. Also download this notebook **with outputs** after the saved run finishes; Kaggle does not place its notebook file in /kaggle/working automatically.

In [ ]:
#!/usr/bin/env python3
"""Collect review outputs without datasets, environments or model weights. Standard library only."""
import argparse
from pathlib import Path
import zipfile
import json
import hashlib

SKIP_DIRS={'.venv','venv','.git','__pycache__','.cache','node_modules','site-packages',
           'patent_runtime_wheels','deberta-v3-small','all-MiniLM-L6-v2','data','adapter',
           'trainer','trainer_tmp','patent_weights','reference_evidence'}
JSON_NAMES={'best_config.json','run_summary.json','test_metrics.json','history.json','CHECKS.json'}
SKIP_FILES={'train.csv','test.csv','sample_submission.csv','IndianFoodDatasetCSV.csv',
            'tokenizer.json','config.json','tokenizer_config.json','adapter_config.json'}
SUFFIXES={'.ipynb','.csv','.png','.jpg','.jpeg','.pdf','.log','.txt'}

def eligible(path):
    if path.is_symlink() or not path.is_file():return False
    if any(p in SKIP_DIRS or p.startswith('checkpoint-') for p in path.parts):return False
    if path.name in SKIP_FILES:return False
    if path.suffix.lower()=='.json':return path.name in JSON_NAMES
    if path.suffix.lower() not in SUFFIXES:return False
    if path.suffix.lower()=='.txt' and path.name not in {'notebook_urls.txt','errors.txt','review_notes.txt'}:return False
    return True

def collect(roots,destination):
    destination=destination.resolve();destination.parent.mkdir(parents=True,exist_ok=True)
    count=0;manifest=[]
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as archive:
        for index,root in enumerate(roots,1):
            root=root.resolve()
            if not root.exists():raise FileNotFoundError(root)
            paths=[root] if root.is_file() else sorted(root.rglob('*'))
            for path in paths:
                if path.resolve()==destination or not eligible(path):continue
                relative=Path(path.name) if root.is_file() else path.relative_to(root)
                arc=Path(f'{index:02d}_{root.stem}')/relative
                archive.write(path,str(arc));count+=1
                manifest.append({'path':str(arc),'bytes':path.stat().st_size,
                                 'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
        archive.writestr('RETURN_MANIFEST.json',json.dumps(manifest,indent=2))
    if count==0:
        destination.unlink();raise ValueError('No review outputs found. Check the input folder.')
    print(f'Saved {count} files: {destination} ({destination.stat().st_size/1e6:.2f} MB)',flush=True)


if Path("/kaggle/working").exists():
    collect([Path("/kaggle/working")],Path("/kaggle/working/A24ARIU0001_UNET_RESULTS.zip"))
